# LitGPT vs HF Transformers: Q/K/V Extraction Benchmark (Llama-3.2-1B)

This notebook runs on Colab (T4 GPU) and compares **HF Transformers** vs **LitGPT** for:
- first 100 samples from Alpaca
- inference with `meta-llama/Llama-3.2-1B-Instruct`
- extraction of Q, K, V at the same attention layer index
- saving Q/K/V tensors to one file per framework
- timing and throughput comparison

Notes:
- You need a Hugging Face token with access to Llama 3.2 weights.
- This benchmark measures prompt forward pass + Q/K/V extraction (not autoregressive generation loops).

In [8]:
!pip install -q litgpt transformers datasets torch huggingface_hub sentencepiece

In [3]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

# ====== User-configurable ======
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
NUM_SAMPLES = 100
LAYER_IDX = 5              # keep identical across both frameworks
MAX_INPUT_TOKENS = 128     # reduce if you hit memory issues
OUT_DIR = Path("qkv_outputs")
STORE_DTYPE = torch.float16
# ===============================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.set_grad_enabled(False)

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        if HF_TOKEN:
            print("HF token loaded from Colab Secrets.")
    except Exception:
        pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Enter your Hugging Face token (hf_...): ").strip()

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF login: OK")
else:
    raise ValueError("HF token is required for this model. Set Colab secret HF_TOKEN or enter it when prompted.")

Device: cuda
GPU: Tesla T4
HF token loaded from Colab Secrets.
HF login: OK


In [4]:
alpaca = load_dataset("tatsu-lab/alpaca", split=f"train[:{NUM_SAMPLES}]")

def build_alpaca_prompt(row):
    instruction = row.get("instruction", "").strip()
    inp = row.get("input", "").strip()
    if inp:
        return f"Instruction:\n{instruction}\n\nInput:\n{inp}\n\nResponse:\n"
    return f"Instruction:\n{instruction}\n\nResponse:\n"

prompts = [build_alpaca_prompt(r) for r in alpaca]
print("Loaded prompts:", len(prompts))
print(prompts[0][:300])

Loaded prompts: 100
Instruction:
Give three tips for staying healthy.

Response:



In [16]:
def summarize_metrics(name, latencies_s, lengths, peak_mem_bytes):
    lat = np.array(latencies_s, dtype=np.float64)
    total_time = float(lat.sum())
    total_tokens = int(np.sum(lengths))
    return {
        "framework": name,
        "samples": int(len(latencies_s)),
        "total_time_s": total_time,
        "mean_latency_ms": float(lat.mean() * 1000.0),
        "p50_latency_ms": float(np.percentile(lat, 50) * 1000.0),
        "p95_latency_ms": float(np.percentile(lat, 95) * 1000.0),
        "prompt_tokens_total": total_tokens,
        "prompt_tokens_per_s": float(total_tokens / total_time) if total_time > 0 else float("nan"),
        "gpu_peak_mem_gb": float(peak_mem_bytes / 1e9) if peak_mem_bytes is not None else float("nan"),
    }

def split_litgpt_qkv(qkv, attn_module):
    # Reconstruct q,k,v exactly as LitGPT does before RoPE is applied
    cfg = attn_module.config
    B, T, _ = qkv.shape
    n_head = int(cfg.n_head)
    n_query_groups = int(getattr(cfg, "n_query_groups", n_head))
    head_size = int(cfg.head_size)

    q_per_kv = n_head // n_query_groups
    total_qkv = q_per_kv + 2

    qkv = qkv.view(B, T, n_query_groups, total_qkv, head_size)
    qkv = qkv.permute(0, 2, 3, 1, 4)
    q, k, v = qkv.split((q_per_kv, 1, 1), dim=2)

    if n_query_groups != n_head:
        q = q.expand(B, n_query_groups, q_per_kv, T, head_size)
        k = k.expand(B, n_query_groups, q_per_kv, T, head_size)
        v = v.expand(B, n_query_groups, q_per_kv, T, head_size)

    q = q.reshape(B, -1, T, head_size).permute(0, 2, 1, 3).reshape(B, T, -1)
    k = k.reshape(B, -1, T, head_size).permute(0, 2, 1, 3).reshape(B, T, -1)
    v = v.reshape(B, -1, T, head_size).permute(0, 2, 1, 3).reshape(B, T, -1)
    return q, k, v

In [17]:
def run_hf(prompts):
    dtype = torch.bfloat16 if DEVICE == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        token=HF_TOKEN,
        torch_dtype=dtype,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE != "cuda":
        model.to(DEVICE)
    model.eval()

    if not (0 <= LAYER_IDX < len(model.model.layers)):
        raise ValueError(f"LAYER_IDX={LAYER_IDX} out of range for HF model ({len(model.model.layers)} layers)")

    attn = model.model.layers[LAYER_IDX].self_attn
    num_heads = int(model.config.num_attention_heads)
    num_kv_heads = int(getattr(model.config, "num_key_value_heads", num_heads))
    head_dim = int(getattr(model.config, "head_dim", model.config.hidden_size // num_heads))
    hidden_size = num_heads * head_dim

    def expand_kv_to_q_heads(x):
        # HF returns grouped K/V for GQA models; expand to query-head width for parity with LitGPT split
        b, t, _ = x.shape
        x = x.view(b, t, num_kv_heads, head_dim)
        if num_kv_heads != num_heads:
            repeat = num_heads // num_kv_heads
            x = x.repeat_interleave(repeat, dim=2)
        return x.reshape(b, t, hidden_size)

    cache = {}
    def mk_hook(name):
        def _hook(_module, _inp, out):
            cache[name] = out.detach()
        return _hook

    handles = [
        attn.q_proj.register_forward_hook(mk_hook("q")),
        attn.k_proj.register_forward_hook(mk_hook("k")),
        attn.v_proj.register_forward_hook(mk_hook("v")),
    ]

    q_store = torch.zeros((NUM_SAMPLES, MAX_INPUT_TOKENS, hidden_size), dtype=STORE_DTYPE)
    k_store = torch.zeros_like(q_store)
    v_store = torch.zeros_like(q_store)
    lengths = []
    latencies = []

    model_device = next(model.parameters()).device

    warm = tokenizer(prompts[0], return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(model_device)
    with torch.inference_mode():
        _ = model(**warm, use_cache=False)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    for i, p in enumerate(prompts):
        cache.clear()
        inputs = tokenizer(p, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(model_device)

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            _ = model(**inputs, use_cache=False)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        q = cache["q"][0].to(STORE_DTYPE).cpu()
        k = expand_kv_to_q_heads(cache["k"])[0].to(STORE_DTYPE).cpu()
        v = expand_kv_to_q_heads(cache["v"])[0].to(STORE_DTYPE).cpu()

        if q.shape[1] != hidden_size or k.shape[1] != hidden_size or v.shape[1] != hidden_size:
            raise ValueError(
                f"Unexpected QKV widths: q={q.shape}, k={k.shape}, v={v.shape}, expected hidden_size={hidden_size}"
            )

        L = min(q.shape[0], k.shape[0], v.shape[0], MAX_INPUT_TOKENS)
        lengths.append(int(L))
        q_store[i, :L] = q[:L]
        k_store[i, :L] = k[:L]
        v_store[i, :L] = v[:L]

    for h in handles:
        h.remove()

    peak_mem = torch.cuda.max_memory_allocated() if DEVICE == "cuda" else None
    out_file = OUT_DIR / f"hf_qkv_layer{LAYER_IDX}.pt"
    torch.save({
        "framework": "hf_transformers",
        "model_id": MODEL_ID,
        "layer_idx": LAYER_IDX,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "kv_expanded_to_num_heads": True,
        "lengths": torch.tensor(lengths, dtype=torch.int32),
        "q": q_store,
        "k": k_store,
        "v": v_store,
    }, out_file)

    metrics = summarize_metrics("hf_transformers", latencies, lengths, peak_mem)
    metrics["qkv_file"] = str(out_file)
    return metrics


In [25]:
from litgpt import LLM
import huggingface_hub.constants
import inspect
if not hasattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER"):
    setattr(huggingface_hub.constants, "HF_HUB_ENABLE_HF_TRANSFER", False)
def run_litgpt(prompts):

    precision = "bf16-true" if DEVICE == "cuda" else "32-true"

    sig = inspect.signature(LLM.load)
    load_kwargs = {}
    if "model" in sig.parameters:
        load_kwargs["model"] = MODEL_ID
    if "distribute" in sig.parameters:
        load_kwargs["distribute"] = "auto"
    if "access_token" in sig.parameters and HF_TOKEN:
        load_kwargs["access_token"] = HF_TOKEN
    if "token" in sig.parameters and HF_TOKEN:
        load_kwargs["token"] = HF_TOKEN

    try:
        llm = LLM.load(**load_kwargs)
    except TypeError:
        # Fallback for older/newer LitGPT signatures
        llm = LLM.load(MODEL_ID)

    if getattr(llm, "model", None) is None:
        if hasattr(llm, "distribute"):
            llm.distribute(
                accelerator="cuda" if DEVICE == "cuda" else "cpu",
                devices=1,
                precision=precision,
            )
        else:
            raise RuntimeError("LitGPT returned model=None and has no distribute() method.")

    if getattr(llm, "model", None) is None:
        raise RuntimeError("LitGPT model is still None after distribute().")

    llm.model.eval()

    if hasattr(llm.model, "transformer") and hasattr(llm.model.transformer, "h"):
        blocks = llm.model.transformer.h
    elif hasattr(llm.model, "layers"):
        blocks = llm.model.layers
    else:
        raise RuntimeError("Unable to locate transformer blocks in LitGPT model")

    if not (0 <= LAYER_IDX < len(blocks)):
        raise ValueError(f"LAYER_IDX={LAYER_IDX} out of range for LitGPT model ({len(blocks)} layers)")

    block = blocks[LAYER_IDX]
    attn = getattr(block, "attn", None) or getattr(block, "self_attn", None)
    if attn is None:
        raise RuntimeError("Unable to locate attention module in LitGPT block")

    qkv_proj = getattr(attn, "attn", None) or getattr(attn, "qkv", None)
    if qkv_proj is None:
        raise RuntimeError("Unable to locate LitGPT qkv projection module for hook")

    cache = {}

    def qkv_hook(_module, _inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        cache["qkv"] = out.detach()

    handle = qkv_proj.register_forward_hook(qkv_hook)

    cfg = llm.model.config
    n_embd = getattr(cfg, "n_embd", None)
    if n_embd is None:
        n_embd = getattr(cfg, "hidden_size", None)
    if n_embd is None:
        n_embd = int(cfg.n_head) * int(cfg.head_size)
    hidden_size = int(n_embd)

    q_store = torch.zeros((NUM_SAMPLES, MAX_INPUT_TOKENS, hidden_size), dtype=STORE_DTYPE)
    k_store = torch.zeros_like(q_store)
    v_store = torch.zeros_like(q_store)
    lengths = []
    latencies = []

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_device = next(llm.model.parameters()).device

    def encode_prompt(text):
        ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS)["input_ids"][0]
        return ids.to(model_device)

    warm_ids = encode_prompt(prompts[0]).unsqueeze(0)
    with torch.inference_mode():
        _ = llm.model(warm_ids)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    class _Wrap:
        pass
    attn_for_split = attn
    if not hasattr(attn_for_split, "config"):
        w = _Wrap()
        w.config = cfg
        attn_for_split = w

    for i, p in enumerate(prompts):
        cache.clear()
        ids = encode_prompt(p)
        x = ids.unsqueeze(0)

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.inference_mode():
            _ = llm.model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        if "qkv" not in cache:
            raise RuntimeError("LitGPT hook did not capture qkv output")

        qkv = cache["qkv"]
        q, k, v = split_litgpt_qkv(qkv, attn_for_split)
        q = q[0].to(STORE_DTYPE).cpu()
        k = k[0].to(STORE_DTYPE).cpu()
        v = v[0].to(STORE_DTYPE).cpu()

        if q.shape[1] != hidden_size or k.shape[1] != hidden_size or v.shape[1] != hidden_size:
            raise ValueError(
                f"Unexpected LitGPT QKV widths: q={q.shape}, k={k.shape}, v={v.shape}, expected hidden_size={hidden_size}"
            )

        L = min(q.shape[0], k.shape[0], v.shape[0], MAX_INPUT_TOKENS)
        lengths.append(int(L))
        q_store[i, :L] = q[:L]
        k_store[i, :L] = k[:L]
        v_store[i, :L] = v[:L]

    handle.remove()

    peak_mem = torch.cuda.max_memory_allocated() if DEVICE == "cuda" else None
    out_file = OUT_DIR / f"litgpt_qkv_layer{LAYER_IDX}.pt"
    torch.save({
        "framework": "litgpt",
        "model_id": MODEL_ID,
        "layer_idx": LAYER_IDX,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "lengths": torch.tensor(lengths, dtype=torch.int32),
        "q": q_store,
        "k": k_store,
        "v": v_store,
    }, out_file)

    metrics = summarize_metrics("litgpt", latencies, lengths, peak_mem)
    metrics["qkv_file"] = str(out_file)
    return metrics


In [21]:
# !pip uninstall -y transformers
# !pip install transformers>=4.43.0

In [12]:
import transformers
print(transformers.__version__)
# Should be >= 4.43.0

5.2.0


In [26]:
hf_metrics = run_hf(prompts)
litgpt_metrics = run_litgpt(prompts)

comparison = pd.DataFrame([hf_metrics, litgpt_metrics])
comparison = comparison[[
    "framework",
    "samples",
    "total_time_s",
    "mean_latency_ms",
    "p50_latency_ms",
    "p95_latency_ms",
    "prompt_tokens_total",
    "prompt_tokens_per_s",
    "gpu_peak_mem_gb",
    "qkv_file",
]]
comparison

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

,framework,samples,total_time_s,mean_latency_ms,p50_latency_ms,p95_latency_ms,prompt_tokens_total,prompt_tokens_per_s,gpu_peak_mem_gb,qkv_file
0,hf_transformers,100,9.020655,90.206551,87.696459,110.259446,2214,245.436721,2.514297,qkv_outputs/hf_qkv_layer5.pt
1,litgpt,100,8.199120,81.991203,79.898276,89.638649,2214,270.028968,3.120647,qkv_outputs/litgpt_qkv_layer5.pt


In [27]:
faster = comparison.sort_values("total_time_s").iloc[0]
print("Faster (this run):", faster["framework"])
print("\nSaved files:")
for p in comparison["qkv_file"].tolist():
    print(" -", p)

print("\nTo load a saved tensor file later:")
print("obj = torch.load('<path>.pt', map_location='cpu')")
print("obj.keys() ->", ["framework", "model_id", "layer_idx", "max_input_tokens", "lengths", "q", "k", "v"])

Faster (this run): litgpt

Saved files:
 - qkv_outputs/hf_qkv_layer5.pt
 - qkv_outputs/litgpt_qkv_layer5.pt

To load a saved tensor file later:
obj = torch.load('<path>.pt', map_location='cpu')
obj.keys() -> ['framework', 'model_id', 'layer_idx', 'max_input_tokens', 'lengths', 'q', 'k', 'v']
